# 📰  Pydantic 파이프라인 + 뉴스 모니터링

```
Phase 0~4. Core 학습 — Pydantic + RunnableParallel 핵심 부품
Project.   기사 → 분석 + 요약 + 태깅 + 대응 라우팅
```

## Phase 0. 환경 설정

In [1]:
%pip install -q langchain langchain-openai python-dotenv pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

✅ 로컬 Jupyter
✅ API 키 OK


## Phase 1. PydanticOutputParser — 자연어를 객체로

5단계 패턴: 스키마 → 파서 → 프롬프트 → LLM → 체인

In [3]:
class Summary(BaseModel):
    title: str = Field(description="짧은 제목")
    sentiment: Literal["positive", "negative", "neutral"]
    key_points: list[str] = Field(description="핵심 2-3개")

parser = PydanticOutputParser(pydantic_object=Summary)

prompt = ChatPromptTemplate.from_messages([
    ("system", "텍스트를 분석합니다. 한국어로.\n\n{format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=parser.get_format_instructions())

chain_summary = prompt | llm | parser

r = chain_summary.invoke({"text": "신제품 출시를 6월에서 7월로 연기하기로 했습니다."})
print(r)

title='신제품 출시 연기' sentiment='neutral' key_points=['신제품 출시 일정 변경', '출시가 6월에서 7월로 연기됨']


## Phase 2. RunnableParallel — 여러 체인 결합

In [4]:
# 자연어 답장 체인 추가 (StrOutputParser)
chain_notice = (
    ChatPromptTemplate.from_messages([
        ("system", "팀원에게 알릴 한 줄 안내문을 작성하세요."),
        ("human", "{text}"),
    ]) | llm | StrOutputParser()
)

combined = RunnableParallel(summary=chain_summary, notice=chain_notice)

r = combined.invoke({"text": "신제품 출시를 6월에서 7월로 연기하기로 했습니다."})
print(r["summary"])
print(r["notice"])

title='신제품 출시 연기' sentiment='neutral' key_points=['신제품 출시 일정 변경', '출시일이 6월에서 7월로 연기됨']
신제품 출시가 6월에서 7월로 연기되었음을 알려드립니다.


## Phase 3. RunnableLambda — 함수도 체인에 (결과는 Pydantic으로)

In [5]:
class Decision(BaseModel):
    urgent: bool
    note: str

def route(x) -> Decision:
    s = x["summary"]
    return Decision(urgent=(s.sentiment == "negative"), note=s.title)

chain_routed = combined | RunnableLambda(route)

d = chain_routed.invoke({"text": "신제품 출시 연기"})
print(d)

urgent=True note='신제품 출시 연기'


## Phase 4. RunnablePassthrough — 원본 보존

In [6]:
full = RunnableParallel(
    summary=chain_summary,
    notice=chain_notice,
    original=RunnablePassthrough(),
)
r = full.invoke({"text": "신제품 출시 연기"})
print(r["original"])  # 입력 그대로 통과

{'text': '신제품 출시 연기'}


---
# 📰 Project. 뉴스 모니터링 + PR 대응 시스템

```
기사 → 분석 + 임원 요약 + 태깅 (동시)
     → 대응 라우팅 (즉시 알림 / 일일 브리핑 / 보관)
```

**모니터링 대상**: NovaTech (가상 AI 스타트업, B2B 챗봇 솔루션)

In [ ]:
SAMPLE = """[테크크런치] AITech, B2B 챗봇 시장 점유율 1위 등극.
LLM 챗봇을 금융·제조 대기업 50여 곳에 공급하며 전년 대비 200% 성장.
미국, 영국 등 주요 시장에서 기업용 폐쇄망 AI 점유율 1위를 기록했다.
다만 일부 고객사는 응답 속도 개선이 필요하다고 지적."""

SAMPLE2 = """[TechCrunch] NovaTech Takes the Lead in the B2B Chatbot Market
NovaTech has secured the top spot in the B2B chatbot market, 
supplying LLM-powered chatbots to more than 50 major companies in the financial services and manufacturing sectors. 
The company reported 200% year-over-year growth.
However, some customers said its chatbots still need faster response times."""

In [45]:
PROFILE = """
회사명: NovaTech
사업 분야: 기업용 생성형 AI 챗봇 및 고객 상담 자동화

제품 및 서비스:
- 기업 웹사이트와 고객센터에서 사용하는 AI 챗봇 제공
- 기업이 보유한 FAQ, 매뉴얼, 정책 문서를 기반으로 답변 생성
- 상담원 연결이 필요한 문의를 분류하고 상담원에게 전달
- 대화 기록과 문의 유형을 분석하는 관리자 대시보드 제공

주요 고객:
- 고객 문의가 많은 중소·중견기업
- 전자상거래, 금융, 교육, IT 서비스 분야의 고객센터 및 상담 조직

고객에게 제공하는 가치:
- 반복 문의 자동 응대로 상담 업무 부담 완화
- 24시간 기본 문의 대응
- 기업 문서에 근거한 일관된 답변 제공
- 문의 유형 및 상담 데이터를 활용한 고객지원 개선

비교 시 확인할 항목:
- 지원하는 언어와 연동 채널
- 답변 정확도 및 출처 표시 기능
- 보안·개인정보 보호와 데이터 저장 정책
- 기존 CRM·상담 시스템과의 연동 범위
- 도입 기간, 가격 모델, 기술 지원 수준

주의:
위 항목 중 기사나 공식 자료에서 확인되지 않은 내용은
자사의 확정 사실로 간주하지 말고 비교 결과에 '근거 부족'으로 표시할 것.
"""

In [ ]:
# 분석 체인 — 자사 관점에서
class NewsAnalysis(BaseModel):
    headline: str = Field(description="15자 이내 헤드라인")
    sentiment: Literal["positive", "negative", "mixed", "neutral"]
    impact_score: int = Field(ge=0, le=10, description="자사 영향도 0-10")
    contains_risk: bool = Field(description="평판/사업 리스크 언급 여부")

p1 = PydanticOutputParser(pydantic_object=NewsAnalysis)
chain_analysis = (
    ChatPromptTemplate.from_messages([
        ("system", "자사(NovaTech, B2B AI 챗봇 스타트업) 관점에서 기사를 분석합니다.\n\n{format_instructions}"),
        ("human", "{article}"),
    ]).partial(format_instructions=p1.get_format_instructions())
    | llm | p1
)
print(chain_analysis.invoke({"article": SAMPLE}))

headline='NovaTech, B2B 챗봇 시장 1위' sentiment='positive' impact_score=8 contains_risk=True


In [8]:
# 임원 보고용 3문장 요약 (자연어)
chain_brief = (
    ChatPromptTemplate.from_messages([
        ("system", "임원 보고용 *정확히 3문장* 요약: 사실 / 자사 관련성 / 시사점."),
        ("human", "{article}"),
    ]) | llm | StrOutputParser()
)
print(chain_brief.invoke({"article": SAMPLE}))

NovaTech는 B2B 챗봇 시장에서 1위로 자리매김하며, 금융 및 제조 대기업 50여 곳에 LLM 챗봇을 공급하여 전년 대비 200% 성장했다. 이는 자사의 기술력과 시장 수요에 대한 적절한 대응을 보여준다. 그러나 고객들의 응답 속도 개선 요구는 향후 서비스 품질 향상에 대한 필요성을 시사한다.


In [9]:
# 태깅 체인 — 다중 선택 (list[Literal])
class NewsTags(BaseModel):
    topics: list[Literal["제품", "재무", "인사", "기술", "규제", "경쟁사", "기타"]]
    teams: list[Literal["PR", "마케팅", "프로덕트", "법무", "재무"]] = Field(
        description="이 기사를 봐야 할 사내 팀"
    )

p2 = PydanticOutputParser(pydantic_object=NewsTags)
chain_tags = (
    ChatPromptTemplate.from_messages([
        ("system", "기사에 적절한 주제와 관련 팀 태그.\n\n{format_instructions}"),
        ("human", "{article}"),
    ]).partial(format_instructions=p2.get_format_instructions())
    | llm | p2
)

# 3개 체인 동시 실행
monitor_v1 = RunnableParallel(
    analysis=chain_analysis,
    brief=chain_brief,
    tags=chain_tags,
)

r = monitor_v1.invoke({"article": SAMPLE})
print(f"📌 {r['analysis'].headline} | 영향도 {r['analysis'].impact_score}")
print(f"🏷  {r['tags'].topics} / 팀: {r['tags'].teams}")

📌 NovaTech, B2B 챗봇 시장 1위 | 영향도 8
🏷  ['기술', '경쟁사'] / 팀: ['마케팅', '프로덕트']


In [10]:
# 대응 라우팅 — RunnableLambda + Pydantic
class PRAction(BaseModel):
    priority: Literal["alert_now", "daily_brief", "archive"]
    channel: Literal["slack_urgent", "slack_daily", "notion"]
    message: str

def route_pr(x) -> PRAction:
    a = x["analysis"]
    if a.sentiment == "negative" and a.impact_score >= 7:
        return PRAction(priority="alert_now", channel="slack_urgent",
                       message=f"🚨 즉시 대응: {a.headline}")
    if a.impact_score >= 4:
        return PRAction(priority="daily_brief", channel="slack_daily",
                       message=f"📋 브리핑: {a.headline}")
    return PRAction(priority="archive", channel="notion",
                   message=f"📁 보관: {a.headline}")

monitor_v2 = monitor_v1 | RunnableLambda(route_pr)
print(monitor_v2.invoke({"article": SAMPLE}))

priority='daily_brief' channel='slack_daily' message='📋 브리핑: NovaTech, B2B 챗봇 시장 1위'


In [11]:
# 여러 기사 batch — 매일 들어오는 수십 건
ARTICLES = [
    {"article": SAMPLE},
    {"article": "NovaTech 일부 고객사 데이터 유출 의혹. 회사는 부인. 보안 업계 우려 제기."},
    {"article": "정부, AI 산업 5조원 지원금 편성. 스타트업·연구기관 대상."},
]
for a in monitor_v2.batch(ARTICLES):
    icon = {"alert_now": "🚨", "daily_brief": "📋", "archive": "📁"}[a.priority]
    print(f"{icon} {a.message}")

📋 📋 브리핑: NovaTech, B2B 챗봇 시장 1위
🚨 🚨 즉시 대응: 고객사 데이터 유출 의혹
📋 📋 브리핑: AI 산업 5조원 지원


---
## 🎓 마무리

같은 3개 부품으로 PR 모니터링 시스템 완성. **도메인이 달라도 패턴은 동일**.

**도전과제**: SNS 카피 체인 / 영문 번역 체인 / 경쟁사 비교 체인 추가

In [ ]:
# 영문 번역 체인
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

translation = ChatPromptTemplate.from_messages([
    ("system", "영문 뉴스 기사를 자연스럽고 정확한 한국어로 번역. 요약하거나 원문에 없는 내용을 추가하지 않을 것."),
    ("human", "{article}"),
]) | llm | StrOutputParser()

article = translation.invoke({"article": SAMPLE2})
print(article)

[TechCrunch] 노바텍, B2B 챗봇 시장에서 선두주자로 자리매김

노바텍이 B2B 챗봇 시장에서 1위를 차지하며, 금융 서비스 및 제조업 분야의 50개 이상의 주요 기업에 LLM 기반 챗봇을 공급하고 있다. 이 회사는 전년 대비 200% 성장했다고 보고했다. 그러나 일부 고객들은 챗봇의 응답 속도가 더 빨라져야 한다고 언급했다.


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel

COMMON_RULE = """
모든 카피에 다음 규칙을 적용하세요.
- '하지만', '그렇지만', '그럼에도'와 같은 역접 표현을 사용하지 마세요.
- 부정적인 표현 대신 긍정적이고 자연스러운 문장으로 작성하세요.
- 1 문장 당 15단어 미만 활용
"""

def copy_chain(platform_instruction):
    prompt = ChatPromptTemplate.from_messages([
        ("system", f"{COMMON_RULE}\n\n{platform_instruction}"),
        ("human", "{article}"),
    ])
    return prompt | llm | StrOutputParser()

chain_copy = RunnableParallel(
    instagram=copy_chain(
        "인스타그램 홍보 카피를 작성하고 해시태그 5개를 포함"
    ),
    facebook=copy_chain(
        "독자에게 자연스럽게 말을 거는 페이스북 게시글을 작성."
    ),
    x=copy_chain(
        "핵심 내용을 담은 X 게시글을 200자 이내로 작성."
    ),
    ad=copy_chain(
        "제품이나 서비스의 가치를 강조하는 광고 카피를 작성하고 행동을 유도."
    ),
)

result = chain_copy.invoke({"article": SAMPLE})
print(result)


{'instagram': 'NovaTech가 B2B 챗봇 시장에서 1위에 올랐습니다. 금융과 제조 대기업 50곳에 공급하며 200% 성장했습니다. 고객의 목소리를 반영해 더욱 빠른 응답 속도를 목표로 합니다. \n\n#NovaTech #챗봇 #B2B #성장 #고객만족', 'facebook': '🎉 NovaTech가 B2B 챗봇 시장에서 1위에 올랐습니다! 🚀\n\nLLM 챗봇을 통해 금융과 제조 대기업 50곳에 공급하며 놀라운 성장을 이뤘습니다. \n\n전년 대비 200% 성장이라는 성과는 정말 대단하네요! \n\n고객의 목소리를 듣고 응답 속도 개선에 힘쓰는 NovaTech의 미래가 기대됩니다. 🌟', 'x': 'NovaTech가 B2B 챗봇 시장에서 1위에 올랐습니다. LLM 챗봇을 50여 개 대기업에 공급하며 전년 대비 200% 성장했습니다. 고객들은 응답 속도 개선을 요청하고 있습니다.', 'ad': 'NovaTech가 B2B 챗봇 시장에서 1위에 올랐습니다. 금융과 제조 대기업 50곳에 공급하며 200% 성장했습니다. 고객의 목소리를 반영해 더욱 빠른 응답 속도를 제공하겠습니다. NovaTech와 함께 혁신적인 경험을 누려보세요.'}


In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

class ComparisonItem(BaseModel):
    topic: str = Field(description="비교 항목")
    competitor: Optional[str] = Field(default=None, description="기사에서 확인한 경쟁사 정보")
    our_company: Optional[str] = Field(default=None, description="프로필에서 확인한 자사 정보")
    result: Literal["competitor_advantage", "our_advantage", "similar", "insufficient_evidence",
    ] = Field(description="근거에 따른 비교 결과")
    evidence: list[str] = Field(description="비교에 사용한 원문 근거")
    takeaway: str = Field(description="자사에 주는 시사점")

class CompetitorComparison(BaseModel):
    competitor_name: Optional[str] = Field(description="기사에서 식별한 경쟁사명")
    comparisons: list[ComparisonItem] = Field(description="비교 항목별 분석")
    overall_summary: str = Field(description="근거에 기반한 전체 요약")

parser = PydanticOutputParser(pydantic_object=CompetitorComparison)

prompt = ChatPromptTemplate.from_template("""
경쟁사 기사{article}를 {company_profile}과 비교 분석하세요.
기사와 프로필에 명시된 사실만 사용하고, 근거가 부족하면 추정하지 마세요.
판단 근거와 시사점을 구분하세요.

{format_instructions}
""").partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser

result = chain.invoke({
    "article": SAMPLE,
    "company_profile": PROFILE,
})

print(result.model_dump())

{'competitor_name': 'NovaTech', 'comparisons': [{'topic': '시장 점유율', 'competitor': 'B2B 챗봇 시장 점유율 1위', 'our_company': '해당 정보 없음', 'result': 'competitor_advantage', 'evidence': ['NovaTech, B2B 챗봇 시장 점유율 1위 등극.'], 'takeaway': '시장 점유율 1위는 고객 신뢰와 브랜드 인지도를 높이는 데 중요한 요소로, 자사도 시장 점유율 확대를 위한 전략이 필요하다.'}, {'topic': '성장률', 'competitor': '전년 대비 200% 성장', 'our_company': '해당 정보 없음', 'result': 'competitor_advantage', 'evidence': ['전년 대비 200% 성장.'], 'takeaway': '상대적으로 높은 성장률을 보이는 경쟁사에 대응하기 위해 자사의 성장 전략을 재검토할 필요가 있다.'}, {'topic': '응답 속도', 'competitor': '일부 고객사가 응답 속도 개선 필요 지적', 'our_company': '해당 정보 없음', 'result': 'insufficient_evidence', 'evidence': ['일부 고객사는 응답 속도 개선이 필요하다고 지적.'], 'takeaway': '응답 속도 개선이 필요하다는 고객 피드백은 자사 서비스 개선의 기회로 삼을 수 있다.'}, {'topic': '고객 지원 개선', 'competitor': '고객지원 개선을 위한 데이터 분석 제공', 'our_company': '문의 유형 및 상담 데이터를 활용한 고객지원 개선', 'result': 'similar', 'evidence': ['대화 기록과 문의 유형을 분석하는 관리자 대시보드 제공', '문의 유형 및 상담 데이터를 활용한 고객지원 개선'], 'takeaway': '고객 지원 개선을 위한 데이터 분석 기능은 자사와 유사하므로, 이를 더욱 